## Test de l'API de scoring Home Credit

Ce notebook permet de :

1. Charger un échantillon de clients depuis le jeu de test.
2. Vérifier que l'API est bien démarrée (`/health`).
3. Envoyer un client à l’endpoint `/predict`.
4. Inspecter la probabilité de défaut et la décision renvoyées par l’API.


## Sommaire

[1. Imports & configuration de base](#1-imports--configuration-de-base)

[2. Chargement du dataset de test et sélection d'un client](#2-chargement-du-dataset-de-test-et-sélection-dun-client)

[3. Vérifier que lAPI répond bien sur health](#3-vérifier-que-lapi-répond-bien-sur-health)

[4. Préparation du payload à envoyer à predict](#4-préparation-du-payload-à-envoyer-à-predict)

[5. Appel à lendpoint predict](#5-appel-à-lendpoint-predict)

[6. Résumé lisible de la prédiction](#6-résumé-lisible-de-la-prédiction)



## 1. Imports & configuration de base

In [19]:
import os
import math
import requests
import pandas as pd

from dotenv import load_dotenv

In [20]:
# Chargement des variables d'environnement (.env à la racine du projet)
load_dotenv()

# URL de base de l'API (par défaut : FastAPI en local via uvicorn)
# Exemple de lancement : uvicorn src.api.app:app --reload
BASE_URL = os.getenv("API_URL", "http://127.0.0.1:8000")

print("BASE_URL =", BASE_URL)

BASE_URL = http://127.0.0.1:8000


## 2. Chargement du dataset de test et sélection d'un client

In [ ]:
# Le fichier correspond à ton X_test imputed (mêmes features que pour l'entraînement)
TEST_PATH = "../data/processed/test_mean_imputed.csv"

df_test = pd.read_csv(TEST_PATH)
print("Shape df_test :", df_test.shape)

# On s'assure qu'il n'y a pas de colonne TARGET dans ce fichier
if "TARGET" in df_test.columns:
    df_test = df_test.drop(columns=["TARGET"])

print("Nombre de features :", df_test.shape[1])

# On sélectionne un client (ici la première ligne)
sample_raw = df_test.iloc[0]
sample_raw.head()

Shape df_test : (48744, 83)
Nombre de features : 83


EXT_SOURCE_3        0.159520
EXT_SOURCE_2        0.789654
EXT_SOURCE_1        0.752614
PAYMENT_RATE        0.036147
DAYS_EMPLOYED   -2329.000000
Name: 0, dtype: float64

## 3. Vérifier que l'API répond bien sur /health

In [ ]:
health = requests.get(f"{BASE_URL}/health")
print("Status code /health :", health.status_code)
print("Réponse /health :")
health.json()

Status code /health : 200
Réponse /health :


{'status': 'ok',
 'model_loaded': True,
 'model_path': 'models/final_lightgbm_model.joblib'}

## 4. Préparation du payload à envoyer à /predict

In [24]:
# On convertit la série en dict
sample_dict = sample_raw.to_dict()

# On remplace les NaN (float) par None pour qu'ils soient sérialisables en JSON
clean_features = {
    k: (None if (isinstance(v, float) and math.isnan(v)) else v)
    for k, v in sample_dict.items()
}

payload = {"features": clean_features}
payload

{'features': {'EXT_SOURCE_3': 0.15951954,
  'EXT_SOURCE_2': 0.7896544,
  'EXT_SOURCE_1': 0.7526145,
  'PAYMENT_RATE': 0.03614715,
  'DAYS_EMPLOYED': -2329.0,
  'AMT_ANNUITY': 20560.5,
  'INSTAL_DPD_MEAN': 1.5714285,
  'CODE_GENDER': 1.0,
  'APPROVED_CNT_PAYMENT_MEAN': 8.0,
  'ACTIVE_DAYS_CREDIT_MAX': -49.0,
  'INSTAL_AMT_PAYMENT_SUM': 41195.926,
  'CC_CNT_DRAWINGS_ATM_CURRENT_MEAN': 0.5662409,
  'OWN_CAR_AGE': 12.061121,
  'ANNUITY_INCOME_PERC': 0.1523,
  'DAYS_ID_PUBLISH': -44.0,
  'NAME_EDUCATION_TYPE_Higher education': 1.0,
  'INSTAL_PAYMENT_PERC_MEAN': 1.0,
  'POS_MONTHS_BALANCE_SIZE': 9.0,
  'INSTAL_DAYS_ENTRY_PAYMENT_MAX': -1628.0,
  'PREV_NAME_CONTRACT_STATUS_Refused_MEAN': 0.0,
  'BURO_DAYS_CREDIT_MEAN': -735.0,
  'BURO_AMT_CREDIT_MAX_OVERDUE_MEAN': 5242.4604,
  'DAYS_REGISTRATION': -5170.0,
  'BURO_AMT_CREDIT_SUM_DEBT_MEAN': 85240.93,
  'ACTIVE_DAYS_CREDIT_ENDDATE_MIN': 411.0,
  'APPROVED_AMT_DOWN_PAYMENT_MAX': 2520.0,
  'POS_SK_DPD_DEF_MEAN': 0.7777778,
  'INCOME_CREDIT_PERC'

## 5. Appel à l'endpoint /predict

In [25]:
response = requests.post(f"{BASE_URL}/predict", json=payload)

print("Status code /predict :", response.status_code)

result = response.json()
result


Status code /predict : 200


{'probability': 0.08723549151078717,
 'prediction': 1,
 'threshold': 0.08406480701683873,
 'model_name': 'LightGBM_final',
 'message': 'Succès de la prédiction.'}

## 6. Résumé lisible de la prédiction


In [26]:
if response.status_code == 200:
    proba = result["probability"]
    pred = result["prediction"]
    thr = result["threshold"]

    decision = "REFUS (défaut probable)" if pred == 1 else "ACCEPTATION (défaut peu probable)"

    print(f"Probabilité de défaut : {proba:.4f}")
    print(f"Seuil métier utilisé  : {thr:.4f}")
    print(f"Classe prédite        : {pred} -> {decision}")
else:
    print("Erreur retournée par l'API :")
    print(result)

Probabilité de défaut : 0.0872
Seuil métier utilisé  : 0.0841
Classe prédite        : 1 -> REFUS (défaut probable)
